# Data Ingestion & Relational Database Pipeline

---

## 📌 Overview
Data ingestion is the foundational stage of this analytics project. This pipeline uses Python to dynamically extract raw, heterogeneous datasets (CSV and Excel files) spread across multiple directories and load them into a unified, structured **MySQL Relational Database Schema**.

Centralizing disjointed raw files into a single relational database establishes a single source of truth, ensures data integrity, and enables scalable SQL querying for downstream exploratory analysis and dashboarding.


In [3]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Load environment variables (defaults to .env in current/root working directory)
load_dotenv(dotenv_path="../.env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# 2. Database Connection
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# 3. Path to Raw Data (using lowercase 'raw')
raw_folder = "../data/raw"

# 4. Hierarchical Ingestion Order (Parent PK tables FIRST, Child FK tables SECOND)
ingestion_order = [
    ("olist_customers_dataset.csv", "olist_customers"),
    ("olist_sellers_dataset.csv", "olist_sellers"),
    ("olist_products_dataset.csv", "olist_products"),
    ("product_category_name_translation.csv", "product_category_translation"),
    ("olist_orders_dataset.csv", "olist_orders"),
    ("olist_order_items_dataset.csv", "olist_order_items"),
    ("olist_order_payments_dataset.csv", "olist_order_payments"),
    ("olist_order_reviews_dataset.csv", "olist_order_reviews"),
    ("olist_geolocation_dataset.csv", "olist_geolocation"),
]

# 5. Ingestion Loop with Exception Handling & Clean Logging
print("🚀 Starting Data Ingestion Pipeline...\n")

for csv_file, table_name in ingestion_order:
    file_path = os.path.join(raw_folder, csv_file)

    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    print("=" * 60)
    print(f"Loading File: {csv_file}")
    print(f"Destination Table: {table_name}")

    try:
        # Load CSV with explicit NA handling
        df = pd.read_csv(file_path, keep_default_na=True)
        print(f"Row Count: {len(df):,}")

        # Append data to pre-existing schema with larger chunksize for performance
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists="append",
            index=False,
            chunksize=10000,
        )
        print(f"✅ Status: SUCCESS - Appended to '{table_name}'")

    except Exception as e:
        print(f"❌ Status: FAILED - Could not append to '{table_name}'")
        print(f"Error Details: {e}")

print("=" * 60)
print("\n🎉 Pipeline Execution Complete!")

ValueError: invalid literal for int() with base 10: 'None'